# 🚀 CareBridge AI - Kaggle GPU Deployment (IMPROVED)

## What This Does:
1. ✅ Checks GPU availability
2. ✅ Clones latest code from GitHub
3. ✅ Installs all dependencies
4. ✅ Downloads MedGemma model (cached for future runs)
5. ✅ Starts FastAPI server
6. ✅ Exposes publicly via ngrok

---

## 🔧 Before Running:

**Kaggle Settings:**
- ⚙️ Accelerator: **GPU T4 x2**
- 🌐 Internet: **ON**
- 💾 Persistence: **Files only**

**Ngrok Setup:**
1. Go to: https://dashboard.ngrok.com/get-started/your-authtoken
2. Sign up (free)
3. Copy your auth token
4. Paste in **Cell 11** (replace `YOUR_NGROK_TOKEN`)

---

## ⏱️ Timing:
- **First run:** ~15-20 minutes (model download)
- **Subsequent runs:** ~3-5 minutes (from cache)

---

## 📝 Cell Order:
**Run cells in order (1 → 12)**

Don't skip cells!

---
## ✅ CELL 1: Check GPU

In [1]:
import subprocess

print("🔍 Checking GPU availability...\n")

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

if result.returncode != 0:
    print('\n❌ ERROR: GPU not found!')
    print('   → Go to: Notebook Settings → Accelerator → Select GPU T4 x2')
    print('   → Then restart kernel\n')
elif 'T4' in result.stdout:
    print('\n✅ Tesla T4 GPU detected - Ready to go!\n')
elif 'A100' in result.stdout:
    print('\n✅ A100 GPU detected - Even better!\n')
else:
    print('\n⚠️  GPU found but not T4/A100')
    print('   → Recommended: Switch to T4 for consistency\n')


🔍 Checking GPU availability...

Sat Apr 11 17:16:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------

---
## 📦 CELL 2: Install System Packages

In [2]:
%%bash
echo "📦 Installing system packages..."
apt-get update -qq
apt-get install -y -qq tesseract-ocr poppler-utils libgl1-mesa-glx > /dev/null 2>&1
echo "✅ System packages installed!"


📦 Installing system packages...
✅ System packages installed!


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


---
## 📂 CELL 3: Clone GitHub Repository

In [5]:
import os
import subprocess

REPO = "https://github.com/AVP2011/CareBridge_AI"
DEST = "/kaggle/working/carebridge"

print("📂 Setting up repository...\n")

if os.path.exists(f"{DEST}/.git"):
    print("   Repository exists - pulling latest changes...")
    result = subprocess.run(
        ["git", "-C", DEST, "pull"],
        capture_output=True,
        text=True
    )
    print(result.stdout)
else:
    print("   Cloning repository...")
    result = subprocess.run(
        ["git", "clone", REPO, DEST],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("   ✅ Clone successful!")
    else:
        print(f"   ❌ Clone failed: {result.stderr}")

print("\n📁 Repository contents:")
subprocess.run(["ls", "-la", DEST])

print("\n✅ Repository ready!")


📂 Setting up repository...

   Repository exists - pulling latest changes...
Updating 23cb21d..153a25c
Fast-forward
 carebridge-ui/package-lock.json                    |   21 +-
 .../src/app/components/layout/Reportchat.tsx       |    9 +
 carebridge-ui/src/app/lib/api.ts                   |   16 +-
 carebridge-ui/src/app/prepurchase/page.tsx         |  110 +-
 carebridge-ui/src/app/types/prepurchase.ts         |   16 +
 engines/pre_purchase_engine.py                     |  349 +-
 .../IRDAIHEALTHINSURANCEREGULATIONS2016.pdf        |  Bin 0 -> 293610 bytes
 ...ter_Circular_on_Protection_of_Policyholders.pdf |  Bin 0 -> 6155716 bytes
 ...aster Circular on Health Insurance Business.pdf |  Bin 0 -> 1602821 bytes
 imp policies/StandardizationOfExclusions2019.pdf   |  Bin 0 -> 1018416 bytes
 imp policies/TPAhhealthserviceRegulations.pdf      |  Bin 0 -> 6839966 bytes
 imp policies/theplanreview.docx                    |  Bin 0 -> 31577 bytes
 llm/prepurchase_prompt.py                       

---
## 🐍 CELL 4: Install Python Dependencies

In [6]:
import subprocess
import sys
!pip install pyngrok
 
BACKEND = "/kaggle/working/carebridge"

print("🐍 Installing Python packages...\n")
print("   This may take 2-3 minutes...\n")

# Install from requirements.txt
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", f"{BACKEND}/requirements.txt", "-q"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ All dependencies installed!\n")
else:
    print(f"⚠️  Some warnings (usually safe to ignore):\n{result.stderr[:500]}\n")

# Verify key packages
print("📋 Verifying key packages:")
packages = ["fastapi", "uvicorn", "transformers", "torch", "pyngrok"]
for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "show", pkg],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        version = [line for line in result.stdout.split('\n') if 'Version:' in line][0]
        print(f"   ✅ {pkg}: {version.split(':')[1].strip()}")
    else:
        print(f"   ❌ {pkg}: NOT FOUND")

print("\n✅ Dependencies ready!")


🐍 Installing Python packages...

   This may take 2-3 minutes...

⚠️  Some warnings (usually safe to ignore):
ERROR: Ignored the following versions that require a different python version: 1.10.0 Requires-Python <3.12,>=3.8; 1.10.0rc1 Requires-Python <3.12,>=3.8; 1.10.0rc2 Requires-Python <3.12,>=3.8; 1.10.1 Requires-Python <3.12,>=3.8; 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 1.6.2 Requires-Python >=3.7,<3.10; 1.6.3 Requires-Python >=3.7,<3.10; 1.7.0 Requires-Python

📋 Verifying key packages:
   ✅ fastapi: 0.133.0
   ✅ uvicorn: 0.41.0
   ✅ transformers: 5.0.0
   ✅ torch: 2.10.0+cu128
   ✅ pyngrok: 8.0.0

✅ Dependencies ready!


---
## 🤖 CELL 5: Check Model Cache

In [8]:
import os

CACHE_DIR = "/kaggle/working/hf_cache"

print("🔍 Checking HuggingFace cache...\n")

if os.path.exists(f"{CACHE_DIR}/hub"):
    print("✅ Model cache exists!")
    print(f"   Location: {CACHE_DIR}/hub")
    
    # Check cache size
    result = subprocess.run(
        ["du", "-sh", f"{CACHE_DIR}/hub"],
        capture_output=True,
        text=True
    )
    print(f"   Size: {result.stdout.split()[0]}")
    print("   → Model will load from cache (faster)\n")
else:
    print("⚠️  No cache found")
    print("   → Model will be downloaded (~4-5 GB)")
    print("   → This will take 10-15 minutes")
    print("   → Future runs will be much faster!\n")

# Create cache directory if doesn't exist
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"✅ Cache directory ready: {CACHE_DIR}")


🔍 Checking HuggingFace cache...

⚠️  No cache found
   → Model will be downloaded (~4-5 GB)
   → This will take 10-15 minutes
   → Future runs will be much faster!

✅ Cache directory ready: /kaggle/working/hf_cache


---
## 🧠 CELL 6: Pre-download Model (Optional but Recommended)

**This downloads the model NOW so server startup is faster.**

Skip this if you want the model to download during server start instead.

In [ ]:
import os
import subprocess
import torch
from huggingface_hub import login, snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM
from kaggle_secrets import UserSecretsClient

# 1. Setup Speed & Storage
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
!pip install -q hf_transfer

print("🧠 Pre-downloading Gemma-2 2B (Optimized Logic)...\n")

# 2. Space Check
print("📊 Checking disk space...")
res = subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True)
print(res.stdout)

# 3. Model Info
MODEL_NAME = "google/gemma-2-2b-it"
print(f"📦 Target Model: {MODEL_NAME}")
print("⚠️  IMPORTANT: Ensure Internet=ON and you accepted the license on HuggingFace.\n")

# 4. Authentication
try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    if hf_token:
        login(hf_token)
        print("✅ HF Auth Successful")
    else:
        print("❌ HF_TOKEN not found in Kaggle Secrets.")
except Exception as e:
    print(f"⚠️  Auth status: {e}")

# 5. Reliable High-Speed Download
print(f"\n📥 Fetching {MODEL_NAME} using hf_transfer...")
try:
    snapshot_download(
        repo_id=MODEL_NAME,
        local_dir_use_symlinks=False,
        cache_dir=os.environ["HF_HOME"]
    )
    print("✅ Download complete!\n")
    
    # Quick Memory Check
    print("🧪 Verifying loading capability...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    print("✅ Model loaded. Clearing memory for server startup...")
    del model
    del tokenizer
    torch.cuda.empty_cache()
    print("🧹 Memory Ready.")
except Exception as e:
    print(f"❌ ERROR: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 38.8 MB/s eta 0:00:00a 0:00:01
🧠 Pre-downloading MedGemma model...

   Model: google/gemma-4b-it
   Size: ~4-5 GB
   Time: 10-15 minutes (first time only)

🔑 Setting up HuggingFace token...
✅ Found HF_TOKEN!
📥 Downloading tokenizer...
   ✅ Tokenizer downloaded

📥 Downloading model (this is the slow part)...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

---
## 🛠️ CELL 7: Check Server Configuration

In [40]:
BACKEND = "/kaggle/working/carebridge"

print("🛠️  Checking server configuration...\n")

# Check if main.py exists
if os.path.exists(f"{BACKEND}/main.py"):
    print("✅ main.py found")
    
    # Show first 30 lines
    print("\n📄 First 30 lines of main.py:\n")
    with open(f"{BACKEND}/main.py", 'r') as f:
        lines = f.readlines()[:30]
        for i, line in enumerate(lines, 1):
            print(f"{i:3d}: {line}", end='')
else:
    print("❌ main.py NOT FOUND!")
    print("   → Check if repo cloned correctly in Cell 3\n")

print("\n✅ Configuration check complete!")


🛠️  Checking server configuration...

✅ main.py found

📄 First 30 lines of main.py:

  1: # main.py
  2: 
  3: import os
  4: from contextlib import asynccontextmanager
  5: 
  6: from fastapi import FastAPI, HTTPException, File, UploadFile
  7: from fastapi.middleware.cors import CORSMiddleware
  8: from pydantic import BaseModel
  9: 
 10: from llm.model_loader import ModelLoader
 11: from engines.post_rejection_engine import PostRejectionEngine
 12: from engines.pre_purchase_engine import PrePurchaseEngine
 13: from engines.policy_comparison_engine import PolicyComparisonEngine
 14: 
 15: from schemas.request import PostRejectionRequest, PrePurchaseRequest
 16: from schemas.chat import ReportChatResponse
 17: from schemas.policy_comparison import PolicyComparisonReport
 18: 
 19: from services.report_chat_service import run_report_chat
 20: from services.chat_memory import create_session
 21: 
 22: import pdfplumber
 23: import pytesseract
 24: from PIL import Image
 25: import io
 

---
## 🚀 CELL 8: Start FastAPI Server

**This starts the server in the background.**

Wait for "Server is ready" message before proceeding.

In [66]:
import subprocess
import time
import os
import requests

BACKEND = "/kaggle/working/carebridge"
CACHE = "/kaggle/working/hf_cache"
LOG_FILE = "/kaggle/working/server.log"

print("🚀 Starting FastAPI Server...\n")

# 1. Safety Cleanup
subprocess.run(["pkill", "-f", "uvicorn"], capture_output=True)
time.sleep(2)

# 2. Environment Setup
env = os.environ.copy()
env.update({
    "PYTHONPATH": BACKEND,
    "HF_HOME": CACHE,
    "HF_HUB_ENABLE_HF_TRANSFER": "1",
    "HUGGINGFACE_HUB_CACHE": f"{CACHE}/hub",
    "TRANSFORMERS_CACHE": f"{CACHE}/hub",
})

# Pass HF_TOKEN to server context
try:
    from kaggle_secrets import UserSecretsClient
    env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except: pass

# 3. Background Startup
with open(LOG_FILE, "w") as f:
    process = subprocess.Popen(
        ["python", "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
        cwd=BACKEND, stdout=f, stderr=subprocess.STDOUT, env=env
    )

print(f"   📝 Logs: {LOG_FILE}")
print("   ⏳ Waiting for initialization...")

# 4. Health Check Loop
for i in range(1, 40):
    try:
        r = requests.get("http://localhost:8000/", timeout=2)
        if r.status_code == 200:
            print(f"\n✅ Server is LIVE! Response: {r.json()}")
            break
    except:
        print(f"   [{i*5}s] still loading...", end="\r")
        time.sleep(5)
else:
    print("\n⚠️  Server timeout. Check logs: !tail -50 /kaggle/working/server.log")

🚀 Starting FastAPI server...

   🧹 Cleaned up old processes

   📝 Server started (PID: 3156)

   ⏳ Waiting for server to be ready...

   (Model loading may take 2-5 minutes)

   [0s] Still loading...
   [10s] Still loading...
   [20s] Still loading...

✅ Server is ready! (took 22 seconds)

📊 Server info:
{'status': 'CareBridge AI v2 running', 'engines': ['post_rejection', 'pre_purchase', 'comparison', 'model', 'tokenizer']}

✅ FastAPI server is LIVE on port 8000!



---
## 📋 CELL 9: Check Server Logs (If Needed)

In [65]:
# Run this if server didn't start properly
!tail -50 /kaggle/working/server.log


INFO:     Started server process [3102]
INFO:     Waiting for application startup.
🔄 CareBridge AI starting up...
🔄 Loading MedGemma 4B in 4-bit mode...
Loading weights: 100%|██████████| 883/883 [00:04<00:00, 182.43it/s, Materializing param=model.vision_tower.vision_model.post_layernorm.weight]                      
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
✅ MedGemma 4B loaded in 4-bit mode
   pad_token_id : 1
   eos_token_id : 1
✅ All engines ready
INFO:     127.0.0.1:43616 - "GET / HTTP/1.1" 200 OK


---
## 🧪 CELL 10: Test Local Server

In [51]:
import requests
import json

print("🧪 Testing local server...\n")

try:
    # Test root endpoint
    response = requests.get("http://localhost:8000/", timeout=10)
    print("✅ Server is responding!\n")
    print("📊 Response:")
    print(json.dumps(response.json(), indent=2))
    print("\n✅ Local server test passed!\n")
except Exception as e:
    print(f"❌ Server test failed: {str(e)}\n")
    print("   → Check server logs in Cell 9\n")


🧪 Testing local server...

✅ Server is responding!

📊 Response:
{
  "status": "CareBridge AI v2 running",
  "engines": [
    "post_rejection",
    "pre_purchase",
    "comparison",
    "model",
    "tokenizer"
  ]
}

✅ Local server test passed!



---
## 🌐 CELL 11: Setup Ngrok Tunnel

**⚠️ IMPORTANT: Replace `YOUR_NGROK_TOKEN` with your actual token!**

Get token from: https://dashboard.ngrok.com/get-started/your-authtoken

In [67]:
from pyngrok import ngrok
import time

# ⚠️⚠️⚠️ REPLACE THIS WITH YOUR ACTUAL NGROK TOKEN! ⚠️⚠️⚠️
NGROK_AUTH_TOKEN = "3AoGH2qgpya7zUT7YyoxAdZQq0i_AXbPAvDyJJP5A6rbGoVV"

if NGROK_AUTH_TOKEN == "YOUR_NGROK_TOKEN_HERE":
    print("❌ ERROR: You need to set your ngrok token!\n")
    print("Steps:\n")
    print("1. Go to: https://dashboard.ngrok.com/get-started/your-authtoken")
    print("2. Sign up (free)")
    print("3. Copy your auth token")
    print("4. Replace 'YOUR_NGROK_TOKEN_HERE' above with your token")
    print("5. Re-run this cell\n")
else:
    print("🌐 Setting up ngrok tunnel...\n")
    
    # Kill any existing tunnels
    ngrok.kill()
    time.sleep(2)
    
    # Set auth token
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    
    # Create tunnel
    try:
        tunnel = ngrok.connect(8000, "http")
        public_url = tunnel.public_url
        
        print("✅ Ngrok tunnel created!\n")
        print("="*60)
        print(f"🌍 PUBLIC URL: {public_url}")
        print("="*60)
        print(f"\n📚 API Documentation: {public_url}/docs")
        print(f"📊 Health Check: {public_url}/")
        print("\n💡 Use this URL in your frontend:\n")
        print(f"   const API_URL = \"{public_url}\"\n")
        print("="*60)
        print("\n⏰ This URL is valid for the current session only")
        print("   (Changes when you restart the notebook)\n")
        
    except Exception as e:
        print(f"❌ Ngrok setup failed: {str(e)}\n")
        print("   → Check if auth token is correct")
        print("   → Verify internet is enabled in Kaggle settings\n")


🌐 Setting up ngrok tunnel...

✅ Ngrok tunnel created!

🌍 PUBLIC URL: https://uninvidious-lyla-capitally.ngrok-free.dev

📚 API Documentation: https://uninvidious-lyla-capitally.ngrok-free.dev/docs
📊 Health Check: https://uninvidious-lyla-capitally.ngrok-free.dev/

💡 Use this URL in your frontend:

   const API_URL = "https://uninvidious-lyla-capitally.ngrok-free.dev"


⏰ This URL is valid for the current session only
   (Changes when you restart the notebook)



---
## ✅ CELL 12: Final Health Check

In [68]:
import requests
import json

if 'public_url' in globals():
    print("🧪 Testing public API...\n")
    
    try:
        response = requests.get(public_url, timeout=10)
        
        if response.status_code == 200:
            print("✅ PUBLIC API IS WORKING!\n")
            print("📊 Server Status:")
            print(json.dumps(response.json(), indent=2))
            print("\n" + "="*60)
            print("🎉 DEPLOYMENT SUCCESSFUL!")
            print("="*60)
            print(f"\n🌍 Your API: {public_url}")
            print(f"📚 Docs: {public_url}/docs")
            print(f"\n💻 Update your frontend with this URL:\n")
            print(f'   const API_URL = "{public_url}"')
            print("\n" + "="*60)
        else:
            print(f"⚠️  API responded with status {response.status_code}")
            
    except Exception as e:
        print(f"❌ Public API test failed: {str(e)}\n")
        print("   → Check if ngrok tunnel is active\n")
else:
    print("⚠️  No public URL found")
    print("   → Run Cell 11 to create ngrok tunnel\n")


🧪 Testing public API...

✅ PUBLIC API IS WORKING!

📊 Server Status:
{
  "status": "CareBridge AI v2 running",
  "engines": [
    "post_rejection",
    "pre_purchase",
    "comparison",
    "model",
    "tokenizer"
  ]
}

🎉 DEPLOYMENT SUCCESSFUL!

🌍 Your API: https://uninvidious-lyla-capitally.ngrok-free.dev
📚 Docs: https://uninvidious-lyla-capitally.ngrok-free.dev/docs

💻 Update your frontend with this URL:

   const API_URL = "https://uninvidious-lyla-capitally.ngrok-free.dev"



---
## 🔄 CELL 13: Restart Server (Without Re-downloading Model)

**Use this if server crashes or you need to restart.**

In [63]:
import subprocess
import time
import os
import requests

BACKEND = "/kaggle/working/carebridge"
CACHE = "/kaggle/working/hf_cache"
LOG_FILE = "/kaggle/working/server.log"

print("🔄 Restarting server...\n")

# Kill existing server
subprocess.run(["pkill", "-f", "uvicorn"], capture_output=True)
time.sleep(3)
print("   🧹 Killed old server\n")

# Set environment
# Set environment variables
env = os.environ.copy()
env.update({
    "PYTHONPATH": BACKEND,
    "HF_HOME": CACHE,
    "HUGGINGFACE_HUB_CACHE": f"{CACHE}/hub",
    "TRANSFORMERS_CACHE": f"{CACHE}/hub",
})

# Make sure HF_TOKEN makes it to the subprocess
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    env["HF_TOKEN"] = token
except Exception:
    token = os.getenv("HF_TOKEN") or os.getenv("HF_TOKENS")
    if token:
        env["HF_TOKEN"] = token

# Start server
log_file = open(LOG_FILE, "w")
server_process = subprocess.Popen(
    ["python", "-m", "uvicorn", "main:app",
     "--host", "0.0.0.0", "--port", "8000", "--workers", "1"],
    cwd=BACKEND,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env
)

print("   ⏳ Waiting for server (model loads from cache)...\n")

# Wait for server
for elapsed in range(10, 240, 10):
    time.sleep(10)
    
    if server_process.poll() is not None:
        print("\n❌ Process died!")
        subprocess.run(["tail", "-30", LOG_FILE])
        break
    
    try:
        r = requests.get("http://localhost:8000/", timeout=5)
        if r.status_code == 200:
            print(f"\n✅ Server restarted! (took {elapsed}s)\n")
            print("   → Re-run Cell 11 to get new ngrok URL\n")
            break
    except:
        print(f"   [{elapsed}s] loading...", end="\r")
else:
    print("\n⚠️  Timeout - check logs: !tail -50 /kaggle/working/server.log")


🔄 Restarting server...

   🧹 Killed old server

   ⏳ Waiting for server (model loads from cache)...

   [20s] loading...
✅ Server restarted! (took 30s)

   → Re-run Cell 11 to get new ngrok URL



---
## 📚 Quick Reference Guide

### Common Tasks:

| Task | What to Do |
|------|------------|
| **First time setup** | Run Cells 1-12 in order |
| **Server crashed** | Run Cell 13 (restart), then Cell 11 (new ngrok) |
| **Check errors** | Run Cell 9 (view logs) |
| **Update code from GitHub** | Run Cell 3 (pull), then Cell 13 (restart) |
| **Get new public URL** | Run Cell 11 |
| **Test if working** | Run Cell 12 |

### Important Files:

- **Backend code:** `/kaggle/working/carebridge/`
- **Model cache:** `/kaggle/working/hf_cache/hub/`
- **Server logs:** `/kaggle/working/server.log`

### Troubleshooting:

**Problem:** Server won't start
- Check logs: `!tail -50 /kaggle/working/server.log`
- Verify GPU is enabled in settings
- Try restarting kernel

**Problem:** Ngrok fails
- Verify auth token is correct
- Check internet is enabled
- Try running Cell 11 again

**Problem:** Out of GPU memory
- Restart kernel
- Ensure no other notebooks are running

### Session Limits:

- **GPU quota:** 30 hours/week (free tier)
- **Session timeout:** 9 hours max
- **Auto-shutdown:** 60 minutes of inactivity

### Production Deployment:

For permanent deployment:
- AWS EC2 with GPU (g4dn.xlarge)
- Google Cloud with T4
- Or upgrade ngrok for static domain ($8/month)

---

## 🎉 You're All Set!

Your CareBridge AI backend is now running on Kaggle GPU!

**Next Steps:**
1. Copy the public URL from Cell 11
2. Update your Next.js frontend with the URL
3. Deploy frontend to Vercel
4. Test end-to-end!

**Need help?** Check the troubleshooting guide above or ask your team! 🚀